# 📊 Notebook 1 — Data Exploration

**Course:** PA2595 Machine Learning Engineering  
**Dataset:** UCI Student Performance (student-mat.csv)

---

## What is this notebook for?

Before we train any machine learning model, we need to **understand our data**. This step is called **Exploratory Data Analysis (EDA)**.

The goal of EDA is to answer questions like:
- How many rows and columns does the dataset have?
- Are there any missing values?
- What do the columns mean?
- How is the target variable (Pass/Fail) distributed?
- Are there any patterns or correlations between features?

> ⚠️ **Important:** Make sure you have downloaded `student-mat.csv` from  
> https://archive.ics.uci.edu/dataset/320/student%2Bperformance  
> and placed it in the `data/raw/` folder before running this notebook.

## Step 1 — Import Libraries

We load the tools we will use throughout this notebook.

- **pandas**: loads and manipulates tabular data (like an Excel spreadsheet in Python)
- **matplotlib** and **seaborn**: create charts and visualisations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots appear inline inside the notebook
%matplotlib inline

# Use a clean visual style for all plots
sns.set_theme(style="whitegrid", palette="muted")

print("Libraries loaded successfully.")

## Step 2 — Load the Dataset

The dataset uses **semicolons (`;`)** as column separators instead of the usual commas.  
We tell pandas this with the `sep=";"` parameter.

In [ ]:
# Load the raw CSV file
df = pd.read_csv("../data/raw/student-mat.csv", sep=";")

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## Step 3 — Column Overview

Let's print a list of all columns with their data types.

- **int64** = integer number (e.g. grade 0–20)
- **object** = text / categorical value (e.g. "yes" / "no", "M" / "F")

In [ ]:
print("Column names and data types:\n")
print(df.dtypes.to_string())

### What does each column mean?

| Column | Type | Description |
|---|---|---|
| `school` | text | School attended (GP or MS) |
| `sex` | text | Student sex (M = Male, F = Female) |
| `age` | number | Student age (15–22) |
| `address` | text | Home address type (U = urban, R = rural) |
| `famsize` | text | Family size (LE3 = ≤3 members, GT3 = >3) |
| `Pstatus` | text | Parents' cohabitation status (T = together, A = apart) |
| `Medu` | number | Mother's education level (0 = none … 4 = higher education) |
| `Fedu` | number | Father's education level (0 = none … 4 = higher education) |
| `traveltime` | number | Home-to-school travel time (1 = <15 min … 4 = >1 hour) |
| `studytime` | number | Weekly study time (1 = <2h … 4 = >10h) |
| `failures` | number | Number of past class failures (0–4) |
| `schoolsup` | text | Extra educational support from school (yes/no) |
| `famsup` | text | Family educational support (yes/no) |
| `paid` | text | Extra paid classes within the course (yes/no) |
| `activities` | text | Extra-curricular activities (yes/no) |
| `nursery` | text | Attended nursery school (yes/no) |
| `higher` | text | Wants to pursue higher education (yes/no) |
| `internet` | text | Internet access at home (yes/no) |
| `romantic` | text | In a romantic relationship (yes/no) |
| `famrel` | number | Quality of family relationships (1 = very bad … 5 = excellent) |
| `freetime` | number | Free time after school (1 = very low … 5 = very high) |
| `goout` | number | Going out with friends (1 = very low … 5 = very high) |
| `Dalc` | number | Workday alcohol consumption (1 = very low … 5 = very high) |
| `Walc` | number | Weekend alcohol consumption (1 = very low … 5 = very high) |
| `health` | number | Current health status (1 = very bad … 5 = very good) |
| `absences` | number | Number of school absences (0–93) |
| `G1` | number | First period grade (0–20) |
| `G2` | number | Second period grade (0–20) |
| `G3` | number | **Final grade (0–20) — this is what we predict** |

## Step 4 — Check for Missing Values

Missing values can cause errors in model training. We check if any column has gaps.

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:\n")
print(missing[missing > 0] if missing.any() else "✅ No missing values found.")

## Step 5 — Basic Statistics

`describe()` gives us a quick statistical summary of all numeric columns:
- **count**: how many non-null values
- **mean**: average value
- **std**: standard deviation (how spread out the values are)
- **min / max**: smallest and largest values
- **25% / 50% / 75%**: quartile values (useful to see the range of "typical" students)

In [ ]:
df.describe().round(2)

## Step 6 — Define the Target Variable (Pass / Fail)

Our model will predict whether a student **passes or fails**.  
We define: **Pass = G3 ≥ 10**, **Fail = G3 < 10**

This is called a **binary classification** problem — there are only two possible outcomes.

In [ ]:
PASS_THRESHOLD = 10

df["target"] = (df["G3"] >= PASS_THRESHOLD).astype(int)

counts = df["target"].value_counts().rename({1: "Pass", 0: "Fail"})
print("Target distribution:")
print(counts)
print(f"\nPass rate: {counts['Pass'] / len(df):.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot(kind="bar", ax=ax, color=["#e74c3c", "#2ecc71"], edgecolor="black")
ax.set_title("Target Distribution: Pass vs Fail", fontsize=14)
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of Students")
ax.set_xticklabels(counts.index, rotation=0)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            str(int(bar.get_height())),
            ha="center", va="bottom", fontsize=12)
plt.tight_layout()
plt.show()

## Step 7 — Distribution of the Final Grade (G3)

Let's look at the raw G3 grade distribution across all students.  
The vertical red line shows the **pass threshold (10)**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["G3"], bins=21, range=(-0.5, 20.5), color="#3498db", edgecolor="black")
ax.axvline(PASS_THRESHOLD - 0.5, color="red", linewidth=2, linestyle="--",
           label=f"Pass threshold (G3 >= {PASS_THRESHOLD})")
ax.set_title("Distribution of Final Grade (G3)", fontsize=14)
ax.set_xlabel("Grade (0-20)")
ax.set_ylabel("Number of Students")
ax.legend()
plt.tight_layout()
plt.show()

## Step 8 — Numeric Feature Distributions

Let's see how the key numeric features are distributed across the dataset.  
This helps us spot skewed data or unusual outliers.

In [ ]:
numeric_features = ["studytime", "absences", "failures", "G1", "G2",
                    "Medu", "Fedu", "traveltime", "freetime", "goout",
                    "Dalc", "Walc", "health", "famrel", "age"]

df[numeric_features].hist(bins=15, figsize=(16, 10), color="#5dade2",
                           edgecolor="black", layout=(3, 5))
plt.suptitle("Distribution of Numeric Features", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Step 9 — Categorical Feature Distributions

Now let's look at the text (categorical) columns and how they split between Pass and Fail.

In [ ]:
categorical_features = ["sex", "address", "famsize", "Pstatus",
                        "schoolsup", "famsup", "paid", "activities",
                        "nursery", "higher", "internet", "romantic"]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    crosstab = pd.crosstab(df[col], df["target"]).rename(columns={0: "Fail", 1: "Pass"})
    crosstab.plot(kind="bar", ax=axes[i], color=["#e74c3c", "#2ecc71"],
                  edgecolor="black", legend=(i == 0))
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis="x", rotation=0)

plt.suptitle("Pass vs Fail by Categorical Feature", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Step 10 — Correlation Heatmap

A **correlation matrix** shows us which numeric features are related to each other and to the target.

- Values close to **+1**: strong positive relationship (both go up together)
- Values close to **-1**: strong negative relationship (one goes up, the other goes down)
- Values close to **0**: no relationship

> 📌 **Key insight to look for:** Which features correlate most strongly with `target`?

In [ ]:
corr_cols = numeric_features + ["G3", "target"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Numeric Features vs Target", fontsize=14)
plt.tight_layout()
plt.show()

## Step 11 — Top Features Correlated with Target

Let's print the features most strongly correlated with `target` (Pass/Fail), ranked from highest to lowest.

In [ ]:
target_corr = corr["target"].drop("target").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
target_corr.plot(kind="barh", ax=ax, color=target_corr.apply(
    lambda x: "#2ecc71" if x > 0 else "#e74c3c"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlation with Target (Pass/Fail)", fontsize=14)
ax.set_xlabel("Pearson Correlation Coefficient")
plt.tight_layout()
plt.show()

print("\nTop 5 most correlated with Pass/Fail:")
print(target_corr.head(5).to_string())

## Step 12 — Study Time vs Pass Rate

Higher study time should lead to a higher pass rate. Let's verify.

In [ ]:
study_pass = df.groupby("studytime")["target"].mean().rename("Pass Rate")
study_labels = {1: "< 2h", 2: "2-5h", 3: "5-10h", 4: "> 10h"}

fig, ax = plt.subplots(figsize=(6, 4))
study_pass.rename(index=study_labels).plot(kind="bar", ax=ax, color="#3498db",
                                            edgecolor="black")
ax.set_title("Pass Rate by Weekly Study Time", fontsize=14)
ax.set_xlabel("Study Time per Week")
ax.set_ylabel("Pass Rate")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## ✅ Summary — Key Findings

Based on this exploration:

1. **Class imbalance:** approximately 2/3 of students pass — the dataset is slightly imbalanced but manageable.
2. **G1 and G2 are the strongest predictors** of final grade G3. This makes sense — prior grades reflect prior performance.
3. **failures** has a strong negative correlation with the target — students with past failures are more likely to fail again.
4. **studytime** has a mild positive correlation — students who study more tend to pass more often.
5. **absences** has a mild negative correlation — more absences slightly increase the risk of failing.
6. **Categorical features** like `higher` (wants higher education = yes) and `internet` show meaningful Pass/Fail splits.

> 📌 **Next step:** Open notebook `02_preprocessing.ipynb` to clean and prepare the data for model training.